# 05. Feature engineering

Phase 3, task 3.2. This notebook shows the features built for the daily forecasts, checks that they only use information available at the forecast origin, and measures the bias correction for forecast wind. The code is in `src/aeron/features/` and is tested in `tests/test_features.py`. Nothing is modelled here.

**Setting.** A forecast is issued at the end of local day O for the daily mean PM2.5 of day D = O + lead (lead 1, 2 or 3 days). A row of the feature table is one origin O. Everything in a row has to be known at the end of day O.

Input: `data/processed/daily_us_embassy.csv` (run the download, cleaning and build scripts first).

In [1]:
from pathlib import Path

import pandas as pd

from aeron.features import (
    FEATURE_SETS,
    bias,
    build_features,
    feature_columns,
    observed_as_forecast,
)

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)

daily = pd.read_csv(
    ROOT / "data" / "processed" / "daily_us_embassy.csv", index_col=0, parse_dates=True
)
LEADS = (1, 2, 3)
tables = {lead: build_features(daily, lead) for lead in LEADS}
frame1 = tables[1]
print(
    f"daily table: {len(daily)} days; feature table at lead 1: {frame1.shape[0]} rows, {frame1.shape[1]} columns"
)

daily table: 2282 days; feature table at lead 1: 2282 rows, 77 columns


## 1. The feature groups

| Group | Prefix | What it holds | Known at the origin? |
|---|---|---|---|
| history | `hist_` | daily PM2.5 of days O, O-1, ..., O-6, rolling 7 and 14 day mean, 7 day standard deviation and maximum, change since yesterday, distance from the 7 day mean, the maximum, evening mean and number of observed hours of day O | yes (looks backwards only) |
| time | `time_` | day of year (sine and cosine), month, winter flag, weekday and weekend flag of the **target** day D | yes (calendar) |
| weather now | `wxo_` | ERA5 temperature, humidity, wind, rain, pressure on day O and day O-1 | yes (observed) |
| weather forecast | `wxf_` | archived weather forecast for day D: temperatures, humidity, wind speed, gust, wind direction as east and north components, rain, pressure, cloud, radiation, vapour pressure deficit | yes (issued before O) |
| weather that happened | `wxt_` | the ERA5 weather of day D, plus boundary layer height and ventilation | **no**: a perfect-forecast scenario for upper bounds only |

Wind direction is a circle, so it enters as east (u) and north (v) components of the wind vector, never as an angle. Cyclical features use sine and cosine.

In [2]:
cols = list(frame1.columns)
sizes = pd.DataFrame(
    {
        "features": [len(feature_columns(cols, s)) for s in FEATURE_SETS],
        "uses perfect-forecast weather": ["no", "no", "no", "no", "yes"],
        "contains boundary layer or snow": [
            any(
                k in c
                for c in feature_columns(cols, s)
                for k in ("blh", "snow", "vent")
            )
            for s in FEATURE_SETS
        ],
    },
    index=FEATURE_SETS,
)
sizes

,features,uses perfect-forecast weather,contains boundary layer or snow
history,22,no,False
history_weather_now,36,no,False
forecast,53,no,False
forecast_no_wind,49,no,False
observed_upper_bound,56,yes,True


## 2. How many rows can each feature set use?

In [3]:
rows = []
for lead, t in tables.items():
    has_y = t["y"].notna()
    has_forecast = t["wxf_temp_mean"].notna()
    rows.append(
        {
            "lead (days)": lead,
            "origins": len(t),
            "with a target": int(has_y.sum()),
            "with target and real forecast weather": int((has_y & has_forecast).sum()),
            "first origin with a forecast": t.index[has_forecast].min().date(),
            "with target and boundary layer": int(
                (has_y & t["wxt_blh_mean"].notna()).sum()
            ),
        }
    )
pd.DataFrame(rows).set_index("lead (days)")

,origins,with a target,with target and real forecast weather,first origin with a forecast,with target and boundary layer
lead (days),,,,,
1,2282,1822,373,2024-01-20,1642
2,2282,1821,372,2024-01-20,1641
3,2282,1820,371,2024-01-20,1640


In [4]:
# Share of missing values by feature group (lead 1, rows that have a target).
t = frame1[frame1["y"].notna()]
groups = {
    "history": "hist_",
    "time": "time_",
    "weather now": "wxo_",
    "weather forecast": "wxf_",
    "weather that happened": "wxt_",
}
pd.DataFrame(
    {
        name: [
            t[[c for c in t.columns if c.startswith(p)]].isna().mean().mean(),
            t[[c for c in t.columns if c.startswith(p)]].isna().any(axis=1).mean(),
        ]
        for name, p in groups.items()
    },
    index=["mean share of missing values", "share of rows with any missing value"],
).T.round(3)

,mean share of missing values,share of rows with any missing value
history,0.080,0.386
time,0.000,0.000
weather now,0.000,0.001
weather forecast,0.795,0.795
weather that happened,0.015,0.099


Archived forecasts exist only from 2024, so features that use them are present for about a year of rows. Models are therefore trained with the weather that happened standing in for the forecast, and tested on the real forecast columns (`observed_as_forecast`, methodology rule 3). The training view has no missing forecast columns apart from the rows where ERA5 itself is missing.

In [5]:
train_view = observed_as_forecast(frame1)
usable_train = train_view["y"].notna() & train_view["wxf_temp_mean"].notna()
usable_test = frame1["y"].notna() & frame1["wxf_temp_mean"].notna()
print(
    f"rows with target and (observed-as-forecast) weather, for training: {int(usable_train.sum())}"
)
print(
    f"rows with target and real forecast weather, for testing: {int(usable_test.sum())}"
)

rows with target and (observed-as-forecast) weather, for training: 1822
rows with target and real forecast weather, for testing: 373


## 3. One row

In [6]:
origin = pd.Timestamp("2024-12-10")
row = frame1.loc[origin]
shown = []
for name, prefix in {
    "time and target": ("origin_date", "target_date", "y", "y_origin", "time_"),
    "history": ("hist_",),
    "weather now": ("wxo_",),
    "forecast for the target day": ("wxf_",),
    "weather that happened on the target day": ("wxt_",),
}.items():
    keys = [k for k in row.index if k.startswith(tuple(prefix))]
    shown.append(pd.DataFrame({"group": name, "value": row[keys]}))
pd.concat(shown).round(2)

,group,value
origin_date,time and target,2024-12-10 00:00:00
target_date,time and target,2024-12-11 00:00:00
y,time and target,99.521739
y_origin,time and target,89.0
time_doy_sin,time and target,-0.325128
time_doy_cos,time and target,0.94567
time_month,time and target,12
time_winter,time and target,1
time_dow,time and target,2
time_weekend,time and target,0


## 4. Leakage check on the real table

The test in `tests/test_features.py` replaces every observation after the origin by garbage and requires that the features known at the origin do not change. The cell below does the same for one origin and prints the largest change per group. The target `y` and the perfect-forecast group `wxt_` must change (they use day D, after the origin), and the groups that are known at the origin must not.

In [7]:
def perturb_after(daily, origin):
    bad = daily.copy()
    after = bad.index > origin
    for col in ["pm25_mean", "pm25_max", "pm25_eve_mean", "pm25_hours"]:
        bad.loc[after, col] = 1e6
    for col in [c for c in bad.columns if c.startswith("era5_")]:
        bad.loc[after, col] = 1e6
    return bad


origin = pd.Timestamp("2024-12-10")
out = []
for lead in LEADS:
    clean = build_features(daily, lead).loc[origin]
    changed = build_features(perturb_after(daily, origin), lead).loc[origin]
    for name, prefix in {
        "history": "hist_",
        "time": "time_",
        "weather now": "wxo_",
        "weather forecast": "wxf_",
        "weather that happened": "wxt_",
    }.items():
        keys = [k for k in clean.index if k.startswith(prefix)]
        diff = (clean[keys].astype(float) - changed[keys].astype(float)).abs().max()
        out.append(
            {
                "lead": lead,
                "group": name,
                "largest change after perturbing the future": diff,
            }
        )
    out.append(
        {
            "lead": lead,
            "group": "target y",
            "largest change after perturbing the future": abs(
                clean["y"] - changed["y"]
            ),
        }
    )
result = pd.DataFrame(out).pivot(
    index="group", columns="lead", values="largest change after perturbing the future"
)
result

lead,1,2,3
group,,,
history,0.000000e+00,0.000000e+00,0.000000e+00
target y,9.999005e+05,9.999185e+05,9.999625e+05
time,0.000000e+00,0.000000e+00,0.000000e+00
weather forecast,0.000000e+00,0.000000e+00,0.000000e+00
weather now,0.000000e+00,0.000000e+00,0.000000e+00
weather that happened,1.000009e+06,1.000010e+06,1.000011e+06


## 5. Which features move with the target? (association only)

Spearman correlation with the target for winter target days, in the training view (observed weather standing in for the forecast, so the whole record can be used). Correlation is not cause and not skill: it only shows which inputs carry a signal worth testing in the models.

In [8]:
def top_associations(lead, n=12):
    t = observed_as_forecast(tables[lead])
    t = t[t["target_date"].dt.month.isin([11, 12, 1, 2]) & t["y"].notna()]
    names = feature_columns(list(t.columns), "forecast")
    names = [c for c in names if t[c].notna().sum() > 100 and t[c].nunique() > 2]
    corr = t[names].corrwith(t["y"], method="spearman")
    top = corr.reindex(corr.abs().sort_values(ascending=False).index).head(n)
    return pd.DataFrame(
        {
            "feature": top.index,
            "spearman": top.values,
            "winter rows": [int(t[c].notna().sum()) for c in top.index],
        }
    ).set_index("feature")


pd.concat({f"lead {lead}": top_associations(lead) for lead in (1, 3)}, axis=1).round(2)

lead 1               lead 3            
                      spearman winter rows spearman winter rows
feature                                                        
hist_day_evening_mean     0.70       661.0      NaN         NaN
hist_pm_lag0              0.63       648.0     0.31       621.0
hist_day_max              0.61       648.0     0.32       621.0
wxf_gust_max             -0.57       689.0    -0.57       687.0
wxf_wind_mean            -0.56       689.0    -0.56       687.0
wxo_wind_mean_l0         -0.48       689.0      NaN         NaN
hist_pm_roll7_max         0.47       646.0     0.34       633.0
hist_pm_roll7_mean        0.46       646.0     0.31       633.0
wxo_msl_mean_l1           0.43       688.0      NaN         NaN
wxf_rain3                -0.42       689.0    -0.42       687.0
wxo_msl_mean_l0           0.42       689.0     0.34       687.0
hist_pm_lag1              0.41       632.0      NaN         NaN
wxf_precip                 NaN         NaN    -0.38       687.0
wxf_temp_min               NaN         NaN    -0.34       687.0
wxf_temp_mean              NaN         NaN    -0.30       687.0
hist_pm_roll14_mean        NaN         NaN     0.30       646.0

## 6. Bias correction of forecast wind

The archived wind forecast is lower than ERA5 on average. The correction multiplies forecast wind by the ratio of mean observed to mean forecast wind, estimated on days up to a cutoff. The cutoff here is 2024-10-31, and the correction is checked on the days after it (November 2024 to February 2025), so the check uses no information from the days it is judged on.

In [9]:
cutoff = pd.Timestamp("2024-10-31")
rows = []
for lead in LEADS:
    forecast = daily[f"fc_lead{lead}_wind_mean"]
    observed = daily["era5_wind_mean"]
    ratio = bias.fit_ratio(forecast, observed, cutoff)
    fit_days = forecast.dropna().index[forecast.dropna().index <= cutoff]
    check_days = forecast.dropna().index[forecast.dropna().index > cutoff]
    fit = bias.bias_report(forecast, observed, ratio, fit_days)
    check = bias.bias_report(forecast, observed, ratio, check_days)
    rows.append(
        {
            "lead": lead,
            "ratio": ratio,
            "fit days": fit["n"],
            "check days": check["n"],
            "bias before (km/h)": check["bias_before"],
            "bias after (km/h)": check["bias_after"],
            "MAE before (km/h)": check["mae_before"],
            "MAE after (km/h)": check["mae_after"],
        }
    )
pd.DataFrame(rows).set_index("lead").round(2)

,ratio,fit days,check days,bias before (km/h),bias after (km/h),MAE before (km/h),MAE after (km/h)
lead,,,,,,,
1,1.51,285,117,-1.55,0.74,1.77,1.47
2,1.51,284,117,-1.57,0.72,1.78,1.50
3,1.57,283,117,-1.78,0.67,1.93,1.64


In [10]:
# Same check for the winter days of the check period only, and the correlation (unchanged by a constant factor).
rows = []
for lead in LEADS:
    forecast, observed = daily[f"fc_lead{lead}_wind_mean"], daily["era5_wind_mean"]
    ratio = bias.fit_ratio(forecast, observed, cutoff)
    days = forecast.dropna().index[forecast.dropna().index > cutoff]
    report = bias.bias_report(forecast, observed, ratio, days)
    both = (
        pd.concat([forecast, observed], axis=1, keys=["f", "o"]).reindex(days).dropna()
    )
    rows.append(
        {
            "lead": lead,
            "n": len(both),
            "correlation with ERA5": both["f"].corr(both["o"]),
            "bias after (km/h)": report["bias_after"],
        }
    )
pd.DataFrame(rows).set_index("lead").round(2)

,n,correlation with ERA5,bias after (km/h)
lead,,,
1,117,0.61,0.74
2,117,0.62,0.72
3,117,0.51,0.67


## 7. Findings

All numbers are from the cells above (processed daily table, US station).

**Feature sets.** There are five: `history` (22 features), `history_weather_now` (36), `forecast` (53, the headline set), `forecast_no_wind` (49) and `observed_upper_bound` (56, the only one that uses weather that happened on the target day and the only one with boundary layer height). The headline sets contain no boundary layer, snow or ventilation feature, because those have no archived forecast.

**Rows.** 1,822 origins have a target at lead 1 (1,821 and 1,820 at leads 2 and 3). Real forecast weather exists for only 373, 372 and 371 of them (from origin 2024-01-20), so models are trained with the observed weather standing in for the forecast (`observed_as_forecast`, 1,822 rows at lead 1) and tested on the real forecast columns. About 39% of rows with a target have at least one missing history feature, because of the data gaps and because rolling statistics need at least 4 of the last 7 days. Trees can take these as missing, and the missing-data policy forbids filling them.

**No leakage.** After replacing every observation after the origin 2024-12-10 by garbage, the history, time, weather-now and forecast groups are unchanged at all three leads (largest change 0), while the target and the perfect-forecast group change by about a million. The same check runs in `tests/test_features.py` on synthetic data and on the real table for several origins, and a test with a deliberately leaky feature (a centred rolling mean) shows that the check can fail.

**What moves with the target.** In winter, with observed weather standing in for the forecast, the strongest associations at lead 1 are the evening mean of the origin day (Spearman 0.70), the origin day's own mean (0.63), forecast maximum gust (-0.57) and wind (-0.56), observed wind at the origin (-0.48), sea-level pressure (+0.42) and 3-day rain (-0.42). At lead 3 the history features weaken to 0.30 to 0.34 while forecast wind keeps -0.56, so by three days the weather of the target day carries more signal than the series' own history. This is an association among inputs and the target and does not say how well a model will do.

**Forecast wind bias.** Fitted on days up to 2024-10-31 (283 to 285 days), the correction factor is 1.51, 1.51 and 1.57 for leads 1, 2 and 3. On the 117 days after the cutoff (November 2024 to February 2025) it changes the wind bias from -1.55, -1.57 and -1.78 km/h to +0.74, +0.72 and +0.67 km/h, and lowers the mean absolute error from 1.77, 1.78 and 1.93 to 1.47, 1.50 and 1.64 km/h (about 17%). The correlation with ERA5 (0.61, 0.62, 0.51) cannot change, so the correction fixes the level and not the timing of windy days. It slightly overcorrects in winter, because the factor was fitted on a period that is mostly the warm months. Whether it helps the PM2.5 forecasts is a question for the models (task 3.4.4), and the factor must always be fitted on days before the block's train cutoff.

**For the next steps**
- Baselines and models (3.3, 3.4) use `forecast` as the headline set, `forecast_no_wind` and the bias-corrected wind variant as comparisons, `history` to show what weather adds, and `observed_upper_bound` as the perfect-forecast ceiling.
- The 3.4.4 ablation of lag windows can be done by dropping `hist_pm_lag*` columns beyond a chosen lag.
